In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.models import run_isolation_forest, run_lof, run_pca, run_random_forest

X = np.load('../data/features/X.npy')
y = np.load('../data/features/y.npy')

print("Shape:", X.shape)
print("Anomalies:", y.sum(), f"({y.mean()*100:.2f}%)")

Shape: (15639, 25)
Anomalies: 698 (4.46%)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

X_train_normal = X_train[y_train == 0]
contamination = float(y.mean())

print("Train:", X_train.shape, "| Train normal only:", X_train_normal.shape)
print("Test:", X_test.shape, "| Anomalies in test:", y_test.sum())
print("Contamination:", round(contamination, 4))

Train: (10947, 25) | Train normal only: (10458, 25)
Test: (4692, 25) | Anomalies in test: 209
Contamination: 0.0446


In [3]:
results = []
results.append(run_isolation_forest(X_train_normal, X_test, y_test, contamination))
results.append(run_lof(X_train_normal, X_test, y_test, contamination))
results.append(run_pca(X_train_normal, X_test, y_test, contamination))

pd.DataFrame(results)

,Model,Precision,Recall,F1,AUC,FPR,TP,FP,FN,TN,TrainTime,InferTime
0,Isolation Forest,0.2255,0.2536,0.2387,0.7431,0.0406,53,182,156,4301,0.279,0.025
1,Local Outlier Factor,0.9262,0.5407,0.6828,0.7695,0.0020,113,9,96,4474,3.129,0.190
2,PCA,0.5787,0.5455,0.5616,0.7604,0.0185,114,83,95,4400,0.033,0.004


In [4]:
results.append(run_random_forest(X_train, y_train, X_test, y_test))

comparison = pd.DataFrame(results).sort_values('F1', ascending=False)
comparison

,Model,Precision,Recall,F1,AUC,FPR,TP,FP,FN,TN,TrainTime,InferTime
3,Random Forest,0.9417,0.5407,0.6869,0.7780,0.0016,113,7,96,4476,0.607,0.031
1,Local Outlier Factor,0.9262,0.5407,0.6828,0.7695,0.0020,113,9,96,4474,3.129,0.190
2,PCA,0.5787,0.5455,0.5616,0.7604,0.0185,114,83,95,4400,0.033,0.004
0,Isolation Forest,0.2255,0.2536,0.2387,0.7431,0.0406,53,182,156,4301,0.279,0.025


In [5]:
comparison.to_csv('../results/comparison.csv', index=False)
print("Saved")

Saved
